## Setup

`nltk` needs three data packages the first time you run this: `punkt`/`punkt_tab` (for the tokenizer)
and `stopwords` (for the stopword list). The cell below downloads them if they're missing.

In [73]:
import os
import re
import html
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download required NLTK data. Safe to re-run: if a package is already
# present, nltk.download just confirms it and moves on (no error).
for pkg in ['punkt', 'punkt_tab', 'stopwords']:
    nltk.download(pkg)

pd.set_option('display.max_colwidth', 120)

[nltk_data] Downloading package punkt to /Users/shawnzzz/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/shawnzzz/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/shawnzzz/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



> **Justing writing this here,so why `keep_default_na=False`:** one of our classes is literally named `None`, and by
> default `pandas.read_csv` treats the string `"None"` as a missing value (NaN). A plain
> `pd.read_csv(...)` silently turns the **entire `None` class (~18k+ rows, our second-largest class)**
> into NaN. Disabling default NA conversion keeps the label intact.

In [75]:
# Resolve paths relative to the repo root 

try:
    REPO_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    REPO_ROOT = os.getcwd()

# If the data folder is not found next to the notebook, also try common fallbacks.
DATA_DIR = os.path.join(REPO_ROOT, 'datasets', 'processed')
if not os.path.isdir(DATA_DIR):
    alt = os.path.expanduser('~/Mental-data-diagnosis-/datasets/processed')
    if os.path.isdir(alt):
        DATA_DIR = alt

INPUT_PATH = os.path.join(DATA_DIR, 'mental_health_cleaned.csv')
print("Reading from:", INPUT_PATH)

# keep_default_na=False so the literal "None" label is NOT read as NaN
df = pd.read_csv(INPUT_PATH, keep_default_na=False, na_values=[])

print(f"Loaded {len(df):,} rows")
print("Columns:", list(df.columns))
df.head()

Reading from: /Users/shawnzzz/Mental-data-diagnosis-/datasets/processed/mental_health_cleaned.csv
Loaded 67,770 rows
Columns: ['text', 'label', 'source']


,text,label,source
0,a question about the third conditional. i was making questions for my students and i ran into a little tricky gramma...,None,Murarka
1,the epitome of my life i've recently requested testing accommodations at my university and was given letters to pers...,ADHD,Murarka
2,what are your favourites offbeat destinations of asia? this is my list. **cambodia** * koh rong: amazing beaches and...,None,Murarka
3,"synesthesia survey (what colour is each month to you?) synesthesia. what is synesthesia? according to google, ""synes...",None,Murarka
4,"science ama series: i’m phil baran, and i’m here to talk about our work at the baran laboratory where we try to simp...",None,Murarka


In [76]:
# Visibility check on labels (should be the 6 shared classes)
print(df['label'].value_counts())

label
Depression    28674
None          21703
Anxiety        6535
Bipolar        5402
ADHD           2961
PTSD           2495
Name: count, dtype: int64


In [77]:
# --- Step 7 resource: English stopwords ---
STOPWORDS = set(stopwords.words('english'))

# --- Compiled patterns ---
URL_RE      = re.compile(r'http\S+|www\.\S+')          # step 3
HTML_TAG_RE = re.compile(r'<[^>]+>')                      # step 4
REDDIT_RE   = re.compile(r'\[removed\]|\[deleted\]', re.IGNORECASE)  # step 4
PUNCT_RE    = re.compile(r"[^a-z0-9'\s]")                # step 5: keep letters, digits, apostrophes, spaces

def preprocess_text(text):
    text = str(text)

    # 2. lowercase
    text = text.lower()

    # 3. remove URLs
    text = URL_RE.sub(' ', text)

    # 4. remove HTML tags + decode HTML entities (&amp;, &#x200b;, ...) + Reddit markup
    text = html.unescape(text)
    text = HTML_TAG_RE.sub(' ', text)
    text = REDDIT_RE.sub(' ', text)

    # 5. remove punctuation / special chars, but KEEP apostrophes for contractions
    text = text.replace('\u2019', "'").replace('\u02bc', "'")  # normalize curly apostrophes
    text = PUNCT_RE.sub(' ', text)

    # 6. tokenize
    tokens = word_tokenize(text)

    # 7. stopword removal (also drop the clitic/punctuation fragments word_tokenize
    #    produces from contractions, e.g. "n't", "'ve", by keeping alphabetic tokens only)
    tokens = [t for t in tokens if t.isalpha() and t not in STOPWORDS]

    return tokens

Quick look at what the function does to a few representative posts:

In [79]:
for i in range(3):
    raw = df['text'].iloc[i]
    print("RAW  :", raw[:140])
    print("CLEAN:", " ".join(preprocess_text(raw))[:140])
    print("-" * 80)

RAW  : a question about the third conditional. i was making questions for my students and i ran into a little tricky grammar. which is correct (foc
CLEAN: question third conditional making questions students ran little tricky grammar correct focusing latter part sentence traveled australia yest
--------------------------------------------------------------------------------
RAW  : the epitome of my life i've recently requested testing accommodations at my university and was given letters to personally give to my profes
CLEAN: epitome life recently requested testing accommodations university given letters personally give professors gave first one saw nice next day 
--------------------------------------------------------------------------------
RAW  : what are your favourites offbeat destinations of asia? this is my list. **cambodia** * koh rong: amazing beaches and a very serene romantic 
CLEAN: favourites offbeat destinations asia list cambodia koh rong amazing beaches serene romantic

In [80]:
df['tokens'] = df['text'].apply(preprocess_text)
df['clean_text'] = df['tokens'].apply(lambda toks: ' '.join(toks))

# Sanity checks
token_counts = df['tokens'].apply(len)
print(f"Rows processed         : {len(df):,}")
print(f"Avg tokens per post    : {token_counts.mean():.1f}")
print(f"Posts empty after clean: {(token_counts == 0).sum()}  (consider dropping before modeling)")
df[["text", "label", "clean_text"]].head()

Rows processed         : 67,770
Avg tokens per post    : 62.3
Posts empty after clean: 120  (consider dropping before modeling)


,text,label,clean_text
0,a question about the third conditional. i was making questions for my students and i ran into a little tricky gramma...,None,question third conditional making questions students ran little tricky grammar correct focusing latter part sentence...
1,the epitome of my life i've recently requested testing accommodations at my university and was given letters to pers...,ADHD,epitome life recently requested testing accommodations university given letters personally give professors gave firs...
2,what are your favourites offbeat destinations of asia? this is my list. **cambodia** * koh rong: amazing beaches and...,None,favourites offbeat destinations asia list cambodia koh rong amazing beaches serene romantic island kratie beautiful ...
3,"synesthesia survey (what colour is each month to you?) synesthesia. what is synesthesia? according to google, ""synes...",None,synesthesia survey colour month synesthesia synesthesia according google synesthesia condition one sense example hea...
4,"science ama series: i’m phil baran, and i’m here to talk about our work at the baran laboratory where we try to simp...",None,science ama series phil baran talk work baran laboratory try simplify way molecules created also macarthur fellow ge...


In [81]:
# Save. clean_text is the column the rest of the pipeline builds on.
OUTPUT_PATH = os.path.join(DATA_DIR, 'mental_health_preprocessed.csv')
df[['text', 'clean_text', 'label', 'source']].to_csv(OUTPUT_PATH, index=False)
print(f"Saved -> {OUTPUT_PATH}")

Saved -> /Users/shawnzzz/Mental-data-diagnosis-/datasets/processed/mental_health_preprocessed.csv
